In [1]:
import numpy as np
from scipy import stats

np.random.seed(42)
weights = np.random.normal(loc=98.5, scale=3.0, size=35)   # грами, 35 пачок
mean = weights.mean()
s = weights.std(ddof=1)          # ddof=1 — вибіркове відхилення, оцінка σ
n = len(weights)
se = s / np.sqrt(n)

# 1. довірчий інтервал для середньої ваги пачки (σ невідоме → t-розподіл)
ci_mean = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
print(mean, s, ci_mean)

# 2. частка пачок, недоважених нижче номіналу 100 г
underweight = weights < 100
p_hat = underweight.mean()
n_p, n_1p = n * p_hat, n * (1 - p_hat)
print(p_hat, n_p, n_1p)             # обидва ≥ 5 → наближення прийнятне
se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop = stats.norm.interval(0.95, loc=p_hat, scale=se_p)
print(ci_prop)

98.10207155329924 2.795611960386001 (np.float64(97.14174524517418), np.float64(99.0623978614243))
0.7714285714285715 27.0 7.999999999999999
(np.float64(0.632313910976657), np.float64(0.9105432318804859))


In [2]:
np.random.seed(3)                       # номер варіанта = 3
JULY = 22.0                             # липень, Одеса (таблиця варіанта 3)
daily_temps = np.random.normal(loc=JULY, scale=2.5, size=30)

np.set_printoptions(precision=3, suppress=True)
print("daily_temps =", daily_temps)
print("mean =", daily_temps.mean())
print("std(ddof=1) =", daily_temps.std(ddof=1))

daily_temps = [26.472 23.091 22.241 17.341 21.307 21.113 21.793 20.432 21.89  20.807
 18.715 24.212 24.203 26.274 22.125 20.988 20.637 18.134 24.456 19.247
 19.037 21.486 25.715 22.592 19.441 20.218 23.563 21.599 20.078 21.425]
mean = 21.68774495307844
std(ddof=1) = 2.312140460678939


In [3]:
n = len(daily_temps)
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)
se = s / np.sqrt(n)

ci_mean = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
t_crit = stats.t.ppf(0.975, df=n - 1)
z_crit = stats.norm.ppf(0.975)

print(f"n = {n}, x̄ = {mean:.4f}, s = {s:.4f}, se = {se:.4f}")
print(f"t_крит(0.975, {n-1}) = {t_crit:.4f}   z_крит = {z_crit:.4f}")
print("95% CI для середньої:", (round(ci_mean[0], 4), round(ci_mean[1], 4)))
print("ширина:", round(ci_mean[1] - ci_mean[0], 4))

# для ілюстрації: що дав би z (неправильно тут)
ci_z = stats.norm.interval(0.95, loc=mean, scale=se)
print("CI через z (некоректно):", (round(ci_z[0], 4), round(ci_z[1], 4)))

n = 30, x̄ = 21.6877, s = 2.3121, se = 0.4221
t_крит(0.975, 29) = 2.0452   z_крит = 1.9600
95% CI для середньої: (np.float64(20.8244), np.float64(22.5511))
ширина: 1.7267
CI через z (некоректно): (np.float64(20.8604), np.float64(22.5151))


In [4]:
above = (daily_temps > JULY).sum()
p_hat = above / n
n_p, n_1p = n * p_hat, n * (1 - p_hat)

print("кількість днів вище 22 °C:", above)
print("p_hat =", round(p_hat, 4))
print("n·p_hat =", n_p, " n·(1−p_hat) =", n_1p)
print("умова np≥5 і n(1−p)≥5 виконана:", (n_p >= 5) and (n_1p >= 5))

se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop = stats.norm.interval(0.95, loc=p_hat, scale=se_p)
print("se_p =", round(se_p, 4))
print("95% CI для частки:", (round(ci_prop[0], 4), round(ci_prop[1], 4)))

print("0.5 всередині інтервалу:", ci_prop[0] <= 0.5 <= ci_prop[1])

кількість днів вище 22 °C: 11
p_hat = 0.3667
n·p_hat = 11.0  n·(1−p_hat) = 19.0
умова np≥5 і n(1−p)≥5 виконана: True
se_p = 0.088
95% CI для частки: (np.float64(0.1942), np.float64(0.5391))
0.5 всередині інтервалу: True


In [5]:
# Перевірка інтерпретації: 5000 повторень тієї самої процедури (n=30, μ=22, σ=2.5) —
# скільки інтервалів накрило істинне середнє.
rng = np.random.default_rng(2024)
B = 5000
hits = 0
for _ in range(B):
    x = rng.normal(loc=JULY, scale=2.5, size=n)
    lo, hi = stats.t.interval(0.95, df=n - 1, loc=x.mean(), scale=x.std(ddof=1) / np.sqrt(n))
    hits += (lo <= JULY <= hi)
print(f"частка інтервалів, що накрили μ=22: {hits / B:.4f}")

частка інтервалів, що накрили μ=22: 0.9484


In [6]:
rows = []
for conf in (0.90, 0.95, 0.99):
    lo, hi = stats.t.interval(conf, df=n - 1, loc=mean, scale=se)
    tq = stats.t.ppf(0.5 + conf / 2, df=n - 1)
    rows.append((f"{int(conf*100)}%", lo, hi, hi - lo, 2 * tq * se, tq))
    print(f"{int(conf*100)}%: ({lo:.4f}, {hi:.4f})  ширина = {hi-lo:.4f}  t_крит = {tq:.4f}")

print()
print("| рівень довіри | нижня межа | верхня межа | ширина | t-критичне |")
print("|---|---|---|---|---|")
for name, lo, hi, w, full, tq in rows:
    print(f"| {name} | {lo:.4f} | {hi:.4f} | {w:.4f} | {tq:.4f} |")

90%: (20.9705, 22.4050)  ширина = 1.4345  t_крит = 1.6991
95%: (20.8244, 22.5511)  ширина = 1.7267  t_крит = 2.0452
99%: (20.5242, 22.8513)  ширина = 2.3271  t_крит = 2.7564

| рівень довіри | нижня межа | верхня межа | ширина | t-критичне |
|---|---|---|---|---|
| 90% | 20.9705 | 22.4050 | 1.4345 | 1.6991 |
| 95% | 20.8244 | 22.5511 | 1.7267 | 2.0452 |
| 99% | 20.5242 | 22.8513 | 2.3271 | 2.7564 |


---

## Контрольні питання

**1. Різниця між точковою оцінкою і довірчим інтервалом.**
Точкова оцінка x̄ = 21.69 °C — це одне число, обчислене з 30 вимірювань. Воно майже напевно не дорівнює
справжньому середньому (адже вибірка випадкова), але воно не повідомляє, наскільки ми можемо помилитися:
число не має ні «запасів», ні оцінки точності. Довірчий інтервал відповідає на друге питання — він
задає діапазон припустимих значень параметра разом із вказаним рівнем довіри, тобто вбудовує в оцінку
інформацію про невизначеність, викликану скінченністю вибірки й невідомим σ.

**2. Чому t, а не z при невідомому σ і невеликому n.**
Нормальний довірчий інтервал `x̄ ± z·σ/√n` вимагає знати σ. Заміна σ на вибіркове s вносить у
стандартизовану величину `(x̄−μ)/(s/√n)` додаткову випадковість, тому вона розподілена як t_{n−1},
а не N(0,1). При n = 30 різниця реальна: t_{0.975,29} = 2.045 проти z = 1.960 — тобто z дав би
інтервал майже на 1.7% вужчий. Використання z з невідомим σ — це систематичне заниження покриття,
тобто «дешевша» впевненість. Коли n велике (свідомо значно більше 30), t_{n−1} візуально збігається з z
і різниця стає нехтуваною, але при n = 30 умова лекції ще не виконана.

**3. Що означає «95% довіри».**
Правильно: якщо цю процедуру (вибірка → інтервал) повторювати нескінченно багато разів на тих самих
даних і за тієї самої генеральної сукупності, то приблизно 95% отриманих інтервалів накриватимуть
істинне значення параметра, а приблизно 5% — ні. Це твердження про **метод і повторюваність**, а не
про випадкову величину «цей інтервал». Після обчислення інтервал — уже фіксоване двозначне число, і
його «ймовірність» не дорівнює 0.95: істинне значення або лежить у ньому, або ні. Поширена фраза
«з ймовірністю 95% істинне середнє лежить у цьому інтервалі» плутає випадкову величину (дані вибірки,
а отже й інтервал) з фіксованим невідомим параметром μ; коректно було б «ми 95% впевнені, що…»
або краще — формулювання через повторення процедури вище.

**4. Три фактори ширини довірчого інтервалу.**
Ширина = `2 · t_{1−α/2, n−1} · s/√n`:
1. **Рівень довіри (коефіцієнт t)** — зростає зі зростанням P (1.699 → 2.045 → 2.756 для 90/95/99%);
   більша довіра ⇒ ширший інтервал, бо треба «схопити» значення з меншою ймовірністю вибірки.
2. **Обсяг вибірки n** — входить як 1/√n; більше n ⇒ вужчий інтервал (з 30 до ≈ 55 днів ширина
   99%-інтервалу зменшиться до ширини поточного 95%-го). Це єдина «ціна», яку можна платити
   без зниження довіри і без зміни генерації.
3. **Розкидання σ (вибіркове s)** — ширина пропорційна s; менша варіативність процесу
   (вужчий розкидання, краща однорідність/стратифікація) ⇒ вужчий інтервал. Тут s = 2.31 °C,
   і саме цей фактор при 30 днях лишається найбільшим «важким» множником.

### Контрольні числові значення для перевірки

| величина | значення |
|---|---|
| x̄ | 21.6877 |
| s (ddof=1) | 2.3121 |
| se | 0.4221 |
| 95% CI для μ (t) | (20.8244, 22.5511) |
| p_hat | 0.3667 (11 днів із 30) |
| 95% CI для p (z) | (0.1942, 0.5391) |